In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def compare_anomaly_models(*model_dfs, id_column='TestKey'):
    df_combined = pd.DataFrame({id_column: model_dfs[0][0][id_column]})
    
    # Voeg resultaten van elk model toe
    number_of_anomalies = 0
    model_names = []
    for df, model_name in model_dfs:
        number_of_anomalies +=1 
        model_names.append(model_name)
        df_combined = df_combined.merge(
            df[[id_column, 'is_anomaly']],
            on=id_column,
            suffixes=('', f'_{model_name}')
        )
        df_combined = df_combined.rename(columns={'is_anomaly': f'{model_name}_anomaly'})
    
    # Tel het aantal anomaliedetecties per datapunt
    anomaly_columns = [col for col in df_combined.columns if col.endswith('_anomaly')]
    df_combined['anomaly_count'] = df_combined[anomaly_columns].sum(axis=1)

    df_combined['anomaly_score'] = df_combined['anomaly_count'] / number_of_anomalies 
    
    return df_combined

## Test 
### import csv's

In [3]:
df_test_same_ans = pd.read_csv('./csv/same_answers_test_checked.csv')
df_test_time_spent = pd.read_csv('./csv/time_spent_test_checked.csv')

# df_clf = pd.read_csv('./csv/clf.csv')
df_copod = pd.read_csv('./csv/copod.csv')
df_ecod = pd.read_csv('./csv/ecod.csv')
df_i_forest = pd.read_csv('./csv/i_forest.csv')
df_lof = pd.read_csv('./csv/lof.csv')
# df_neighbours = pd.read_csv('./csv/neighbours.csv')

In [4]:
# time spent + same answers 
df_test_anomalies = df_test_same_ans.merge(df_test_time_spent)
df_test_anomalies.head()

,TestKey,SameAnswerPercentage,TimeSpent,TooSlow,TooFast
0,160065,30.769231,2915,True,False
1,160066,33.333333,2048,False,False
2,160067,28.205128,0,False,False
3,160068,30.769231,0,False,False
4,160069,28.205128,3962,True,False


### Compare and anomaly score

In [5]:
df_test_same_ans['is_anomaly'] = df_test_same_ans.SameAnswerPercentage >= 50
df_test_same_ans.is_anomaly = df_test_same_ans.is_anomaly.astype(int)
df_test_same_ans.head()

,TestKey,SameAnswerPercentage,is_anomaly
0,160065,30.769231,0
1,160066,33.333333,0
2,160067,28.205128,0
3,160068,30.769231,0
4,160069,28.205128,0


In [6]:
df_test_time_spent['is_anomaly'] = df_test_time_spent.TooFast | df_test_time_spent.TooSlow
df_test_time_spent.is_anomaly = df_test_time_spent.is_anomaly.astype(int)
df_test_time_spent.head()

,TestKey,TimeSpent,TooSlow,TooFast,is_anomaly
0,160065,2915,True,False,1
1,160066,2048,False,False,0
2,160067,0,False,False,0
3,160068,0,False,False,0
4,160069,3962,True,False,1


In [7]:
df_compare = compare_anomaly_models(
    # (df_clf, 'clf'),
    (df_copod, 'copod'), 
    (df_ecod, 'ecod'), 
    (df_i_forest, 'i_forest'),
    (df_lof, 'lof'),
    (df_test_same_ans, 'same_ans'),
    (df_test_time_spent, 'timespent'),
    # (df_neighbours, 'neighbours'),
    id_column='TestKey'
)
df_compare.head()

,TestKey,copod_anomaly,ecod_anomaly,i_forest_anomaly,lof_anomaly,same_ans_anomaly,timespent_anomaly,anomaly_count,anomaly_score
0,160065,0,0,0,0,0,1,1,0.166667
1,160066,0,0,0,0,0,0,0,0.000000
2,160067,0,0,0,0,0,0,0,0.000000
3,160068,0,0,0,0,0,0,0,0.000000
4,160069,0,0,0,0,0,1,1,0.166667


In [8]:
df_compare.to_csv('./csv/compare.csv', index=False)

In [9]:
df_compare.head()

,TestKey,copod_anomaly,ecod_anomaly,i_forest_anomaly,lof_anomaly,same_ans_anomaly,timespent_anomaly,anomaly_count,anomaly_score
0,160065,0,0,0,0,0,1,1,0.166667
1,160066,0,0,0,0,0,0,0,0.000000
2,160067,0,0,0,0,0,0,0,0.000000
3,160068,0,0,0,0,0,0,0,0.000000
4,160069,0,0,0,0,0,1,1,0.166667


In [10]:
df_test_anomalies['TooFast'] = df_test_anomalies.TooFast.astype(int)
df_test_anomalies['TooSlow'] = df_test_anomalies.TooSlow.astype(int)

In [11]:
df_test_anomalies = df_compare.merge(df_test_time_spent[['TestKey', 'TooSlow', 'TooFast']], on='TestKey')
df_test_anomalies = df_test_anomalies.merge(df_test_same_ans[['TestKey', 'SameAnswerPercentage']], on='TestKey')
df_test_anomalies.drop(columns=['timespent_anomaly', 'same_ans_anomaly'], inplace=True)
df_test_anomalies.head()

,TestKey,copod_anomaly,ecod_anomaly,i_forest_anomaly,lof_anomaly,anomaly_count,anomaly_score,TooSlow,TooFast,SameAnswerPercentage
0,160065,0,0,0,0,1,0.166667,True,False,30.769231
1,160066,0,0,0,0,0,0.000000,False,False,33.333333
2,160067,0,0,0,0,0,0.000000,False,False,28.205128
3,160068,0,0,0,0,0,0.000000,False,False,30.769231
4,160069,0,0,0,0,1,0.166667,True,False,28.205128


In [12]:
df_test_anomalies.to_csv("../../decoded_data/SJT/DimTestAnomalies.csv", index=False)

In [13]:
# df_test_anomalies = df_test_anomalies.merge(df_compare[['TestKey', 'anomaly_score']], on='TestKey')
# df_test_anomalies

### Store in DimTestAnomalies

In [14]:
# df_test_anomalies.set_index("TestKey", inplace=True)
# df_test_anomalies.to_csv("../../decoded_data/SJT/DimTestAnomalies.csv")
# df_test_anomalies.head()

## QUESTION 
# import csvs


In [15]:
df_question_same_ans = pd.read_csv('./csv/same_answers_question_checked.csv')
df_question_time_spent = pd.read_csv('./csv/time_spent_question_checked.csv')


# same answer + time spent 


In [16]:
df_question_anomalies = df_question_time_spent.merge(df_question_same_ans)


# store in DimQuestionAnomalies


In [17]:
df_question_anomalies.set_index("QuestionKey", inplace=True)
df_question_anomalies.to_csv('../../decoded_data/SJT/DimQuestionAnomalies.csv')